In [2]:
import os

import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline, Pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model_id = "openai/whisper-large-v3"


def create_pipeline() -> Pipeline:
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id, dtype=dtype, low_cpu_mem_usage=True, use_safetensors=True
    )
    model.to(device)

    processor = AutoProcessor.from_pretrained(model_id)

    generation_config = {
        "num_beams": 1,
        "condition_on_prev_tokens": False,
        "compression_ratio_threshold": 1.35,  # zlib compression ratio threshold (in token space)
        "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
        "logprob_threshold": -1.0,
        "no_speech_threshold": 0.6,
        "return_timestamps": True,
    }

    return pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        dtype=dtype,
        device=device,
        config=generation_config,
        return_timestamps=True
    )


## parses text from file and returns:
# (
#     filename,
#     {
#         test: string,
#         chunks: {
#             timestamp: (number, number),
#             text: string
#         }[]
#     }
# )[]
def annotate_data(source_folder: str) -> list[tuple[str, object]]:
    sources = os.listdir(source_folder)

    print("Found files:")
    print(sources)

    pipe = create_pipeline()

    data = []

    for source in sources:
        source_path = os.path.join(source_folder, source)
        source_annotation = pipe(source_path)
        data.append([source_path, source_annotation])

    return data

def generate_augmented_dataset(source_folder: str) -> list[tuple[str, object]]:
    annotated_data = annotate_data(source_folder) # filename + annotation

    # TODO:
    # for each entry:
    # load file with librosa
    # split file by chunks from annotation and contain in array
    #

# accepts (filename_base, librosa audio file object, parsed text)
def augment_dataset(item):
    # TODO:
    # yield every augmented variand (don't forget empty augmentation)
    pass

# TODO
# Then save annotated entries to corresponding files and create index file

# annotate_data("./sonya_source")

Found files:
['Sonya_1.wav', 'Sonya_2.wav']


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

KeyboardInterrupt: 